# Lab 03 · Prompt chaining with quality gates

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## What is prompt chaining?
A prompt chain breaks a task into smaller model calls. Python saves the answer from one call and inserts it into the next prompt. The notebook sets the order of the calls, so you can see what each call receives and produces.

First ask for a lunch idea, then ask for its ingredients. Next use the same approach to compare ways of dividing ShopEasy's checkout system and write down the decision. A complete run makes five model calls.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Step 1 · A simple chain: lunch → ingredients
Imagine you want a lunch idea and then a shopping list.

1. Ask the model to suggest one lunch. Python saves its answer in `lunch`.
2. The `f` before the second prompt string tells Python to replace `{lunch}` with that answer. The model then lists three ingredients for that dish.

If the first call suggests vegetable pasta, the second call gets that exact dish name. Compare the two labeled outputs. Your answer may differ, but the ingredients should fit the lunch you actually got.

In [ ]:
lunch = ask("Suggest one simple vegetarian lunch. Name the dish only.")
print("1. Lunch:", lunch)

ingredients = ask(f"The lunch is: {lunch}\nList three ingredients, one per line.")
print("2. Ingredients:\n", ingredients)

## Next: apply the chain to ShopEasy

ShopEasy sells jackets. A customer places an order, stock is reserved, payment is authorized, and a parcel is shipped. Order, Inventory, and Shipping currently share tables and releases. The next example uses several model calls to explore possible boundaries.

**At a glance**

![Lab 03 at a glance: Brief, then Business capabilities (3 labels), then gate, then Boundary options (2 options · 3 criteria), then Trade-offs (+ / − table), then gate, then Decision record (Context · Decision · Consequence), then gate.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/03.svg)

## Think about the example

A technical-layer split places UI, business logic, and data access in different services. A bounded context defines where one business model and language apply; its deployment boundary still needs review.

**Concepts:** Event storming; Bounded contexts and ubiquitous language; Business capability vs technical-layer decomposition.

Use the scenario above to consider these questions before running the code:

1. Why can a technical-layer split create a distributed monolith?
2. Why is a bounded context a service candidate rather than an automatic deployment decision?

## The idea in context

A fixed prompt chain exposes intermediate results. Gates can catch a malformed handoff while leaving domain judgement for review.

A bounded context is a model boundary; validate it before choosing deployment boundaries.

| Key term | Meaning |
| --- | --- |
| **Workflow** | A sequence of steps fixed by code; the model does not choose what happens next. |
| **Decomposition** | Splitting a large task into smaller steps with clear inputs and outputs. |
| **Gate** | A plain-code check between two steps that stops a bad result from spreading. |
| **Decision record (ADR)** | A short note of an architecture decision: its context, the decision, and its consequence. |

## Pattern · Single-Path Planning

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Passive Goal Creator.*

| | |
| --- | --- |
| **Context** | Complex goals need several dependent steps, where each result informs the next. |
| **Problem** | A single response hides the intermediate choices a later step depends on. |
| **Solution** | Make one ordered plan of a few steps, execute each step with the accumulated context, then synthesize the outputs. |
| **Use when** | The work is naturally sequential; steps depend on earlier outputs; one reasonable path is enough. |
| **Trade-off** | Easy to follow, but an early planning mistake propagates through the whole run. |
| **Flow** | Plan → Step 1 → Step N → Synthesize |
| **Quality attributes** | Benefits: Predictability, Performance, Observability, Maintainability. Liabilities: Reliability, Latency, Usability, Adaptability. |
| **In this lab** | Events lead to candidate bounded contexts, then a comparison of boundary strategies and a short decision record. Gates check output structure; domain experts still validate the proposed boundaries. |

## Step 2 · Prepare the ShopEasy example
Run this cell to store the checkout events, two ways to divide the system, and the criteria for comparing them. An **event** names something that has happened, such as `OrderPlaced`.

A **business capability split** groups work by what the business does, such as taking orders or shipping parcels. A **technical layer split** groups work by how software is built, such as screens, business logic and data access. Steps 3–5 use the events to propose boundaries, compare those alternatives, and record a decision. All the output in this cell comes from Python.

In [ ]:
import json

brief = ("ShopEasy sells jackets. A customer places an order; stock is reserved; payment is authorized; "
         "a parcel is shipped. Order, Inventory and Shipping currently share tables and releases.")
events = ["OrderPlaced", "StockReserved", "PaymentAuthorized", "ParcelShipped"]
options = ["business capability split", "technical layer split"]
criteria = ["cohesive business rules", "independent change", "private data ownership"]
print("Events:", events)
print("Options:", options)

contexts, tradeoffs, record = None, None, None

def read_json(reply):
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        raise ValueError("Quality gate: expected JSON without extra text or code fences.") from None

## Step 3 · Call 1: suggest bounded contexts
A **bounded context** is a part of the business where terms and rules have a consistent meaning. For example, Shipping may need a parcel's weight while Inventory needs the quantity in stock. A context is a candidate boundary to review before deciding how to deploy services.

This prompt gives the model the brief and four checkout events. It asks for three labels because an external provider handles payment in this example. Expect a reply shaped like `["Orders", "Inventory", "Shipping"]`. The JSON parser reads the reply as a Python list, then the **quality gate**, a check between steps, verifies that it contains exactly three nonempty text labels. Read the labels yourself to see whether they describe sensible areas of business work.

In [ ]:
contexts, tradeoffs, record = None, None, None
prompt = (f"Brief: {brief}\nEvents: {events}\nPropose exactly 3 business bounded contexts for "
          "OrderPlaced, StockReserved and ParcelShipped. Payment is supplied externally. "
          "Reply with a JSON list of context labels only, no code fences or explanation.")
reply = ask(prompt)
print("Contexts reply:", reply)
labels = read_json(reply)
if not isinstance(labels, list) or len(labels) != 3:
    raise ValueError("Quality gate: expected a JSON list of three context labels.")
if any(not isinstance(label, str) or not label.strip() for label in labels):
    raise ValueError("Quality gate: each context label must be nonempty text.")
contexts = labels

## Step 4 · Call 2: compare boundary choices
The next prompt includes the brief and the actual `contexts` list from Step 3. It asks for a row for each boundary option and a `+` or `-` for each of the three criteria. For example, `+` for **independent change** means the model thinks that option would let teams change their part with less coordination.

Expect six marks in a nested JSON object that Python reads into dictionaries. The gate checks that the reply contains exactly the requested rows and marks. Read them as the model's judgments about this scenario.

In [ ]:
tradeoffs, record = None, None
if contexts is None:
    raise ValueError("Quality gate: complete Step 3 before comparing boundaries.")
prompt = (f"Brief: {brief}\nContexts: {contexts}\nCompare {options} using {criteria}. "
          'JSON only: {"<option>": {"<criterion>": "+" or "-"}}. Marks are qualitative, not measured.')
reply = ask(prompt)
print("Trade-offs reply:", reply)
comparison = read_json(reply)
if not isinstance(comparison, dict) or set(comparison) != set(options):
    raise ValueError("Quality gate: expected one row per boundary option.")
for option in options:
    marks = comparison[option]
    if not isinstance(marks, dict) or set(marks) != set(criteria):
        raise ValueError(f"Quality gate: expected one mark per criterion for {option}.")
    for criterion, mark in marks.items():
        if mark not in ("+", "-"):
            raise ValueError(f"Quality gate: invalid mark for {option} / {criterion}.")
tradeoffs = comparison

## Step 5 · Call 3: record a decision and its cost
The final prompt includes the same brief, `contexts` and `tradeoffs`. It asks for a short **architecture decision record (ADR)** with three labeled lines.

- **Context** describes the problem that needs a decision.
- **Decision** names one of the two boundary options.
- **Consequence** names a cost of that option, such as more published interfaces to maintain.

The gate requires exactly three labeled lines with text and checks that the decision names an allowed option. Compare the reported cost with the scenario and the earlier trade-offs.

In [ ]:
record = None
if contexts is None or tradeoffs is None:
    raise ValueError("Quality gate: complete Steps 3 and 4 before recording a decision.")
prompt = (f"Brief: {brief}\nContexts: {contexts}\nTrade-offs: {json.dumps(tradeoffs)}\n"
          "Write exactly Context:, Decision:, Consequence: lines. Decision is one option label only. "
          "Consequence must name a cost; do not assume the service candidates are already deployed.")
reply = ask(prompt)
print("Decision record:\n", reply)
fields = [line.partition(":") for line in reply.strip().splitlines()]
if [name for name, _, _ in fields] != ["Context", "Decision", "Consequence"]:
    raise ValueError("Quality gate: expected exactly Context, Decision, Consequence lines.")
if any(not separator or not value.strip() for _, separator, value in fields):
    raise ValueError("Quality gate: each labeled line needs text after the colon.")
if fields[1][2].strip().rstrip(".").lower() not in options:
    raise ValueError("Quality gate: Decision must be one of the two listed options.")
record = reply

## What you should see
Look for a lunch with three ingredients, then three context labels, six trade-off marks and a three-line decision record. Each call prints its reply so you can see where a later result came from.

If a quality gate fails, read the printed reply and its error message. Extra prose or code fences can make the reply invalid JSON. Rerun the failed cell, then continue in order. Each rerun clears results from later steps, so they must use the new reply.

## Try it
1. Make a gate fire on purpose. In Step 3, change the check to expect four entries while the prompt still requests three. Rerun Step 3 and read the message. Restore the check and rerun Step 3 successfully before continuing.
2. Consider a reply with `UI`, `BusinessLogic`, and `DataAccess`. It has three entries and passes the format check. Explain why it might still be a poor way to divide the business.
3. Explain why `Product` has different fields in Catalog, Inventory and Shipping when Catalog uses title and price, Inventory tracks quantity, and Shipping tracks package weight.
4. A customer returns a damaged jacket. After inspection it is refunded and then restocked or scrapped. Which language, rules and data would you investigate before drawing another boundary?

## Takeaway
A chain makes intermediate results visible. Gates check the shape of each handoff, and people review whether the proposed boundaries make sense.